In [2]:
import requests
import urllib3

# Disable SSL warning (temporary fix for network environments)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 28.6,
    "longitude": 77.2,
    "daily": "temperature_2m_max",
    "forecast_days": 7
}

response = requests.get(url, params=params, timeout=10, verify=False)

print(f"Status code: {response.status_code}")
print(f"URL that was called: {response.url}")

Status code: 200
URL that was called: https://api.open-meteo.com/v1/forecast?latitude=28.6&longitude=77.2&daily=temperature_2m_max&forecast_days=7


In [3]:
# Parse the JSON response
data = response.json()

# Look at the top level keys
print("Top level keys:", list(data.keys()))

# Look at what's inside daily
print("\nDaily keys:", list(data['daily'].keys()))

# Look at the actual values
print("\nDates:", data['daily']['time'])
print("Temps:", data['daily']['temperature_2m_max'])

Top level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']

Daily keys: ['time', 'temperature_2m_max']

Dates: ['2026-03-10', '2026-03-11', '2026-03-12', '2026-03-13', '2026-03-14', '2026-03-15', '2026-03-16']
Temps: [33.6, 33.7, 31.5, 29.3, 30.5, 31.0, 31.4]


In [4]:
import pandas as pd

# Extract just the data we need
daily_data = data['daily']

# Convert to DataFrame
df = pd.DataFrame({
    'date': daily_data['time'],
    'temp_max': daily_data['temperature_2m_max']
})

# Set proper data types
df['date'] = pd.to_datetime(df['date'])

print(df)
print(f"\nDtypes:\n{df.dtypes}")

        date  temp_max
0 2026-03-10      33.6
1 2026-03-11      33.7
2 2026-03-12      31.5
3 2026-03-13      29.3
4 2026-03-14      30.5
5 2026-03-15      31.0
6 2026-03-16      31.4

Dtypes:
date        datetime64[us]
temp_max           float64
dtype: object


In [5]:
import requests
import urllib3
import logging

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def fetch_weather(latitude: float, longitude: float, days: int = 7) -> dict:
    """
    Fetch daily max temperature forecast from Open-Meteo API.
    
    Args:
        latitude: Location latitude.
        longitude: Location longitude.
        days: Number of forecast days (default 7).
    
    Returns:
        Raw API response as dictionary.
    
    Raises:
        requests.exceptions.HTTPError: If API returns 4xx/5xx status.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max",
        "forecast_days": days
    }
    
    try:
        response = requests.get(url, params=params, timeout=10, verify=False)
        response.raise_for_status()  # raises HTTPError for 4xx/5xx
        logger.info(f"Successfully fetched weather for ({latitude}, {longitude})")
        return response.json()
    
    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP error: {e}")
        raise
    except requests.exceptions.ConnectionError as e:
        logger.error(f"Connection error - check your network: {e}")
        raise
    except requests.exceptions.Timeout:
        logger.error(f"Request timed out for ({latitude}, {longitude})")
        raise

# Test it
data = fetch_weather(28.6, 77.2)
print(f"Got {len(data['daily']['time'])} days of data")

2026-03-10 10:39:34,118 - INFO - Successfully fetched weather for (28.6, 77.2)


Got 7 days of data


In [6]:
# Test what happens with a bad URL
try:
    bad_data = fetch_weather(999, 999)  # invalid coordinates
except Exception as e:
    print(f"Caught: {type(e).__name__}: {e}")

2026-03-10 10:53:22,024 - ERROR - HTTP error: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&forecast_days=7


Caught: HTTPError: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&forecast_days=7


In [7]:
import time

def fetch_weather_with_retry(latitude: float, longitude: float, 
                              days: int = 7, retries: int = 3) -> dict:
    """
    Fetch weather data with automatic retry on failure.

    Args:
        latitude: Location latitude.
        longitude: Location longitude.
        days: Number of forecast days.
        retries: Number of retry attempts before giving up.

    Returns:
        Raw API response as dictionary.

    Raises:
        Exception: If all retry attempts fail.
    """
    for attempt in range(retries):
        try:
            logger.info(f"Attempt {attempt + 1} of {retries}")
            data = fetch_weather(latitude, longitude, days)
            return data  # success — exit immediately
        
        except Exception as e:
            logger.warning(f"Attempt {attempt + 1} failed: {e}")
            
            if attempt == retries - 1:
                logger.error(f"All {retries} attempts failed. Giving up.")
                raise
            
            wait = 2 ** attempt  # 1s, 2s, 4s — exponential backoff
            logger.info(f"Waiting {wait}s before retry...")
            time.sleep(wait)

# Test with good coordinates
data = fetch_weather_with_retry(28.6, 77.2)
print(f"Success — got {len(data['daily']['time'])} days")

2026-03-10 10:54:28,512 - INFO - Attempt 1 of 3
2026-03-10 10:54:28,785 - INFO - Successfully fetched weather for (28.6, 77.2)


Success — got 7 days


In [8]:
import os

def parse_to_dataframe(raw: dict) -> pd.DataFrame:
    """
    Parse raw Open-Meteo API response into a clean DataFrame.

    Args:
        raw: Raw API response dictionary.

    Returns:
        Cleaned DataFrame with date and temp_max columns.
    """
    df = pd.DataFrame({
        'date': raw['daily']['time'],
        'temp_max': raw['daily']['temperature_2m_max']
    })
    df['date'] = pd.to_datetime(df['date'])
    logger.info(f"Parsed DataFrame: {df.shape[0]} rows, {df.shape[1]} columns")
    return df


def save_to_parquet(df: pd.DataFrame, path: str) -> None:
    """
    Save DataFrame to Parquet file.

    Args:
        df: DataFrame to save.
        path: Full file path including filename.

    Returns:
        None
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    size_kb = os.path.getsize(path) / 1024
    logger.info(f"Saved to {path} ({size_kb:.2f} KB)")


# --- Run the full pipeline ---
raw = fetch_weather_with_retry(28.6, 77.2)
df = parse_to_dataframe(raw)
save_to_parquet(df, "../data/processed/delhi_weather.parquet")

print("\nFinal DataFrame:")
print(df)

2026-03-10 10:56:15,162 - INFO - Attempt 1 of 3
2026-03-10 10:56:16,012 - INFO - Successfully fetched weather for (28.6, 77.2)
2026-03-10 10:56:16,012 - INFO - Parsed DataFrame: 7 rows, 2 columns
2026-03-10 10:56:16,080 - INFO - Saved to ../data/processed/delhi_weather.parquet (1.83 KB)



Final DataFrame:
        date  temp_max
0 2026-03-10      33.6
1 2026-03-11      33.7
2 2026-03-12      31.5
3 2026-03-13      29.3
4 2026-03-14      30.5
5 2026-03-15      31.0
6 2026-03-16      31.4


In [9]:
import json

def save_raw_json(raw: dict, path: str) -> None:
    """
    Save raw API response as JSON. Never modify raw data.

    Args:
        raw: Raw API response dictionary.
        path: Full file path including filename.

    Returns:
        None
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(raw, f, indent=2)
    logger.info(f"Raw response saved to {path}")

# Save raw
save_raw_json(raw, "../data/raw/delhi_weather_raw.json")

2026-03-10 11:03:20,070 - INFO - Raw response saved to ../data/raw/delhi_weather_raw.json
